# 🛕 Heritage Temple Damage — MoE Ensemble (Ultra‑Fast)
### Mixture-of-Experts: ResNet50 · EfficientNet‑B4 · ViT‑B16 · YOLO‑damage
**Goal**: <1 min per expert on T4 GPU | **Fallback chain**: gate → ensemble → mock
**Outputs**: damage location heatmap (Grad‑CAM) + criticality grade (STABLE/MINOR/MODERATE/CRITICAL)

---
## 🚀 Speed Optimizations
- 5 epochs total (1 frozen + 4 fine‑tune)
- Gradient accumulation (effective batch 128)
- `torch.compile()` (~20% faster)
- Modern `torch.amp` (no deprecation warnings)
- Persistent workers & prefetching


In [ ]:
%%capture
!pip install ultralytics timm torch-ema albumentations scikit-learn seaborn grad-cam --quiet

import os, sys, random, json, shutil, time, platform, gc, warnings
from pathlib import Path
from collections import defaultdict
from contextlib import contextmanager

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
import torchvision.transforms as T
import cv2
import timm
from torch_ema import ExponentialMovingAverage
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from tqdm.auto import tqdm
from PIL import Image as PILImage, ImageFile
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

# --- Robustness ---
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", ".*Premature end of JPEG.*")
warnings.filterwarnings("ignore", ".*Corrupt EXIF.*")

# --- Performance ---
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True

assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime > T4 GPU"
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
print(f"Torch: {torch.__version__}")


## ⚙️ Section 1 — Configuration (Ultra‑Fast)

In [ ]:
# --- CONFIGURATION (speed optimized) ---
# Kaggle credentials (read from environment, never hardcode)
KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
KAGGLE_KEY      = os.environ.get("KAGGLE_KEY", "")

GLOBAL_SEED     = 42
BASE_DIR        = Path("/kaggle/working")

# --- Class schema
CLASS_NAMES     = ["Undamaged", "Partial Damage", "Damaged"]
NUM_CLASSES     = len(CLASS_NAMES)
CRITICALITY_MAP = {
    "Undamaged":      "STABLE",
    "Partial Damage": "MINOR",
    "Damaged":        "CRITICAL",
}

# --- Expert training (VERY FAST) ---
IMAGE_SIZE        = 224
BATCH_SIZE        = 32
GRAD_ACCUM_STEPS  = 4                     # effective batch = 128
NUM_WORKERS       = 4
EXPERT_EPOCHS     = 5                     # total epochs (1 stage1 + 4 stage2)
STAGE1_EPOCHS     = 1                     # frozen backbone
STAGE2_EPOCHS     = 4                     # fine‑tune
WARMUP_EPOCHS     = 1                     # linear warmup
LR_HEAD           = 1e-3
LR_LAYER4         = 2e-5
LR_MIN            = 1e-7
WEIGHT_DECAY      = 1e-4
LABEL_SMOOTHING   = 0.1
GRAD_CLIP         = 1.0
EMA_DECAY         = 0.9998
MIXUP_ALPHA       = 0.4
DROPOUT           = 0.4
EARLY_STOP_PAT    = 2                     # stop if no improvement for 2 epochs
COMPILE_MODEL     = True                  # torch.compile (~20% speedup)

# --- Gating network training ---
GATE_EPOCHS     = 5
GATE_LR         = 3e-4
GATE_HIDDEN     = 256

# --- Inference ---
CONF_THRESHOLD  = 0.70
NUM_EXPERTS     = 4

# --- Directories ---
DIRS = {
    "data":     BASE_DIR / "data",
    "raw":      BASE_DIR / "raw",
    "weights":  BASE_DIR / "weights",
    "outputs":  BASE_DIR / "outputs",
    "gradcam":  BASE_DIR / "gradcam",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

DATA_DIR    = DIRS["data"]
WEIGHTS_DIR = DIRS["weights"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Global timing tracker
TIMINGS: dict[str, float] = {}

print(f"Device          : {DEVICE}")
print(f"Classes         : {CLASS_NAMES}")
print(f"Effective batch : {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Total epochs    : {EXPERT_EPOCHS} (Stage1: {STAGE1_EPOCHS}, Stage2: {STAGE2_EPOCHS})")
print(f"torch.compile   : {COMPILE_MODEL}")


In [ ]:
def seed_everything(seed=GLOBAL_SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()

@contextmanager
def timer(name: str):
    t0 = time.perf_counter()
    print(f"\n>>> {name} ...")
    try:
        yield
    finally:
        elapsed = time.perf_counter() - t0
        TIMINGS[name] = elapsed
        m, s = divmod(elapsed, 60)
        print(f">>> {name} done -- {int(m)}m {s:.1f}s")

def save_checkpoint(state, name):
    path = WEIGHTS_DIR / name
    tmp = path.with_suffix(".tmp")
    torch.save(state, tmp)
    tmp.rename(path)
    print(f"  Saved -> {path.name}")

def load_checkpoint(name):
    path = WEIGHTS_DIR / name
    if path.exists():
        return torch.load(path, map_location=DEVICE, weights_only=False)
    return None

def validate_image(path):
    try:
        img = PILImage.open(path)
        img.verify()
        return True
    except Exception:
        return False

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
print("Utilities ready")


## 📥 Section 2 — Dataset Download (with extra relevant datasets)

In [ ]:
with timer("Dataset download"):
    kaggle_dir = Path("/root/.kaggle")
    kaggle_dir.mkdir(exist_ok=True)
    cred_file = kaggle_dir / "kaggle.json"
    if not cred_file.exists() or cred_file.stat().st_size < 10:
        if KAGGLE_KEY:
            with open(cred_file, "w") as f:
                json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
            os.chmod(str(cred_file), 0o600)
            print("  Wrote kaggle.json from environment")
        else:
            print("  No Kaggle key found — relying on pre‑existing credentials")

    RAW = DIRS["raw"]

    # Original + extra datasets (more building/heritage damage)
    DATASETS = [
        ("arnavr10880/concrete-crack-images-for-classification", "concrete_cracks"),
        ("arunrk7/surface-crack-detection",                      "surface_cracks"),
        ("anasmohammedtahir/aider",                              "aider"),
        ("varpit94/disaster-images-dataset",                     "disaster"),
        ("doken/quakeset",                                       "quakeset"),
        ("mohamedhany07/building-damage-dataset",                "building_damage"),
        ("priyankundu/building-damage-assessment",               "building_damage2"),
        ("salmandotkar/building-damage-dataset",                 "building_damage3"),
    ]

    for dataset_id, folder in DATASETS:
        dest = RAW / folder
        if dest.exists() and sum(1 for _ in dest.rglob("*") if _.is_file()) > 10:
            n = sum(1 for _ in dest.rglob("*") if _.is_file())
            print(f"  SKIP  {folder:<35} ({n} files)")
            continue
        dest.mkdir(parents=True, exist_ok=True)
        ret = os.system(f"kaggle datasets download -d {dataset_id} -p {dest} --unzip -q")
        n   = sum(1 for _ in dest.rglob("*") if _.is_file())
        status = "OK" if ret == 0 and n > 0 else "FAIL"
        print(f"  {status}  {folder:<35} ({n} files)")
        if ret != 0 or n == 0:
            print(f"       Dataset {dataset_id} unavailable — continuing anyway.")


## 🗂️ Section 3 — Organise into train/val/test (skip corrupt images)

In [ ]:
with timer("Dataset organisation"):
    seed_everything()
    SPLITS = {"train": 0.70, "val": 0.15, "test": 0.15}

    for split in SPLITS:
        for cls in CLASS_NAMES:
            (DATA_DIR / split / cls).mkdir(parents=True, exist_ok=True)

    # Skip if already organised
    existing = sum(1 for _ in (DATA_DIR / "train").rglob("*") if _.is_file())
    if existing > 100:
        print(f"  SKIP -- already have {existing} training images")
    else:
        RAW = DIRS["raw"]
        SOURCES = [
            (RAW / "disaster/Comprehensive Disaster Dataset(CDD)/Non_Damage/Non_Damage_Buildings_Street",
                                                                      "Undamaged",       None),
            (RAW / "concrete_cracks/Negative",                         "Undamaged",       1500),
            (RAW / "surface_cracks/Positive",                          "Partial Damage",  4000),
            (RAW / "concrete_cracks/Positive",                         "Partial Damage",  4000),
            (RAW / "disaster/Comprehensive Disaster Dataset(CDD)/Damaged_Infrastructure",
                                                                      "Damaged",         None),
            (RAW / "quakeset",                                         "Damaged",         2000),
            (RAW / "aider",                                            "Damaged",         2000),
            (RAW / "building_damage",                                  "Damaged",         2000),
            (RAW / "building_damage2",                                 "Damaged",         2000),
            (RAW / "building_damage3",                                 "Damaged",         2000),
        ]

        def collect_images(folder, cap):
            imgs = [p for p in Path(folder).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
            if cap and len(imgs) > cap:
                imgs = random.sample(imgs, cap)
            return imgs

        def copy_split(images, cls, src_tag):
            random.shuffle(images)
            n = len(images)
            n_train = int(n * SPLITS["train"])
            n_val   = int(n * SPLITS["val"])
            buckets = (("train", images[:n_train]),
                       ("val",   images[n_train:n_train+n_val]),
                       ("test",  images[n_train+n_val:]))
            skipped = 0
            for split, imgs in buckets:
                for i, img in enumerate(imgs):
                    if not validate_image(img):
                        skipped += 1
                        continue
                    dst = DATA_DIR / split / cls / f"{src_tag}_{i:05d}{img.suffix}"
                    shutil.copy2(img, dst)
            if skipped:
                print(f"    Skipped {skipped} corrupt images from {src_tag}")

        for folder, cls, cap in SOURCES:
            folder = Path(folder)
            if not folder.exists():
                print(f"  MISSING {folder.name} -- skipping")
                continue
            imgs = collect_images(folder, cap)
            if not imgs:
                print(f"  EMPTY  {folder.name} -- 0 images found")
                continue
            tag = folder.name[:12].replace(" ", "_")
            copy_split(imgs, cls, tag)
            print(f"  {cls:<18} <- {folder.name[:40]} ({len(imgs)} images)")

    # --- Class distribution & imbalance warning
    print("\nClass distribution:")
    class_counts = {}
    for split in SPLITS:
        for cls in CLASS_NAMES:
            n = len(list((DATA_DIR / split / cls).glob("*")))
            class_counts.setdefault(cls, 0)
            class_counts[cls] += n
            print(f"  {split:6s} / {cls:<18}: {n:5d}")

    max_cls = max(class_counts.values())
    min_cls = min(class_counts.values())
    if max_cls > 3 * min_cls:
        ratio = max_cls / max(min_cls, 1)
        print(f"\n⚠️ SEVERE CLASS IMBALANCE: {ratio:.1f}:1 ratio!")
        print("   WeightedRandomSampler will compensate, but consider adding more data.")


## 🔧 Section 4 — Transforms & Dataset (robust)

In [ ]:
sz   = IMAGE_SIZE
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)

train_tfm = A.Compose([
    A.RandomResizedCrop(size=(sz, sz), scale=(0.65, 1.0), ratio=(0.75, 1.33)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.Affine(rotate=(-20, 20), translate_percent=(-0.05, 0.05), scale=(0.9, 1.1), shear=(-10, 10), p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    A.CLAHE(clip_limit=3.0, p=0.35),
    A.RandomShadow(p=0.2),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.25),
    A.OneOf([A.MotionBlur(blur_limit=5), A.GaussianBlur(blur_limit=5)], p=0.2),
    A.ImageCompression(quality_range=(75, 95), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 6), hole_height_range=(8, sz//10), hole_width_range=(8, sz//10), fill=0, p=0.25),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

val_tfm = A.Compose([
    A.Resize(sz, sz),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

class DamageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
        self.samples = []
        for cls in CLASS_NAMES:
            for p in (Path(root) / cls).iterdir():
                if p.suffix.lower() in IMG_EXTS:
                    self.samples.append((str(p), self.class_to_idx[cls]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        except Exception:
            img = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label

def get_dataloaders():
    train_ds = DamageDataset(DATA_DIR / "train", train_tfm)
    val_ds   = DamageDataset(DATA_DIR / "val",   val_tfm)
    test_ds  = DamageDataset(DATA_DIR / "test",  val_tfm)

    labels = [s[1] for s in train_ds.samples]
    counts = np.bincount(labels)
    weights = 1.0 / counts[labels]
    sampler = WeightedRandomSampler(weights, len(weights))

    loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=3)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, **loader_kw)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **loader_kw)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **loader_kw)

    print(f"Train: {len(train_ds):5d} | Val: {len(val_ds):5d} | Test: {len(test_ds):5d}")
    print(f"Batches/epoch: {len(train_loader)} (eff batch = {BATCH_SIZE * GRAD_ACCUM_STEPS})")
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_dataloaders()


## 🏗️ Section 5 — Expert Model Builders

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.ls = label_smoothing
        self.w = weight
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.w, label_smoothing=self.ls, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

def build_resnet50():
    m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
    for p in m.parameters(): p.requires_grad = False
    m.fc = nn.Sequential(
        nn.BatchNorm1d(2048), nn.Dropout(DROPOUT),
        nn.Linear(2048, 512), nn.ReLU(inplace=True),
        nn.BatchNorm1d(512), nn.Dropout(0.3),
        nn.Linear(512, NUM_CLASSES),
    )
    return m

def build_efficientnet_b4():
    m = tvm.efficientnet_b4(weights=tvm.EfficientNet_B4_Weights.IMAGENET1K_V1)
    for p in m.parameters(): p.requires_grad = False
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(DROPOUT),
        nn.Linear(in_f, 512), nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, NUM_CLASSES),
    )
    return m

def build_vit_b16():
    m = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
    for p in m.parameters(): p.requires_grad = False
    embed_dim = m.embed_dim
    head = nn.Sequential(
        nn.LayerNorm(embed_dim),
        nn.Linear(embed_dim, 256), nn.GELU(),
        nn.Dropout(DROPOUT),
        nn.Linear(256, NUM_CLASSES),
    )
    m.head = head
    for block in m.blocks[-4:]:
        for p in block.parameters(): p.requires_grad = True
    for p in m.head.parameters(): p.requires_grad = True
    return m

class YOLOFeatureClassifier(nn.Module):
    def __init__(self, yolo_weights_path=None):
        super().__init__()
        self.feature_dim = 256  # fixed for fallback CNN
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, NUM_CLASSES),
        )
    def forward(self, x):
        feats = self.backbone(x)
        feats = feats.flatten(1)
        return self.head(feats)

def build_yolo_damage():
    return YOLOFeatureClassifier()

EXPERT_BUILDERS = {
    "resnet50":        build_resnet50,
    "efficientnet_b4": build_efficientnet_b4,
    "vit_b16":         build_vit_b16,
    "yolo_damage":     build_yolo_damage,
}
print("Expert builders defined:")
for name in EXPERT_BUILDERS:
    print(f"  - {name}")


## 🏋️ Section 6 — Expert Training Loop (fast)

In [ ]:
def get_optimizer(model, stage):
    if stage == 1:
        params = [p for p in model.parameters() if p.requires_grad]
        return torch.optim.AdamW(params, lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
    try:
        groups = [
            {"params": model.layer4.parameters(), "lr": LR_LAYER4},
            {"params": model.layer3.parameters(), "lr": LR_LAYER4 / 2},
        ]
        head_p = list(model.fc.parameters()) if hasattr(model, "fc") else list(model.head.parameters())
        groups.append({"params": head_p, "lr": LR_HEAD})
    except AttributeError:
        groups = [{"params": [p for p in model.parameters() if p.requires_grad], "lr": LR_HEAD * 0.1}]
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

def unfreeze_all(model):
    for p in model.parameters(): p.requires_grad = True

def mixup_batch(images, labels, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(images.size(0)).to(DEVICE)
    return lam * images + (1 - lam) * images[idx], labels, labels[idx], lam

def evaluate_expert(model, ema, loader):
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0
    criterion = FocalLoss(gamma=2.0, label_smoothing=LABEL_SMOOTHING)
    with ema.average_parameters():
        with torch.no_grad():
            for imgs, lbls in loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                lbls = lbls.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    out = model(imgs)
                    loss = criterion(out, lbls)
                total_loss += loss.item() * len(imgs)
                all_preds.extend(out.argmax(1).cpu().tolist())
                all_labels.extend(lbls.cpu().tolist())
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    f1 = f1_score(all_labels, all_preds, average="weighted")
    return total_loss / len(all_labels), acc, f1, all_preds, all_labels

def train_expert(name, model, resume=True):
    print(f"\n{'='*60}\n  Training expert: {name}\n{'='*60}")
    ckpt_best = f"{name}_best.pth"
    ckpt_latest = f"{name}_latest.pth"
    model = model.to(DEVICE)

    # Compile if requested
    compiled = model
    if COMPILE_MODEL:
        try:
            compiled = torch.compile(model, mode="reduce-overhead")
            print(f"  torch.compile enabled")
        except Exception as e:
            print(f"  torch.compile skipped: {e}")

    ema = ExponentialMovingAverage(model.parameters(), decay=EMA_DECAY)
    criterion = FocalLoss(gamma=2.0, label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler("cuda")

    best_f1 = 0.0
    start_epoch = 0
    stage = 1
    patience = 0
    history = defaultdict(list)

    # Class weights
    labels = [s[1] for s in train_loader.dataset.samples]
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    weights = torch.tensor(1.0 / (counts + 1e-6), dtype=torch.float32).to(DEVICE)
    weights = weights / weights.sum() * NUM_CLASSES
    criterion.w = weights

    if resume:
        ckpt = load_checkpoint(ckpt_latest)
        if ckpt:
            model.load_state_dict(ckpt["model_state"])
            ema.load_state_dict(ckpt["ema_state"])
            best_f1 = ckpt["best_f1"]
            start_epoch = ckpt["epoch"] + 1
            stage = ckpt.get("stage", 1)
            history = defaultdict(list, ckpt.get("history", {}))
            print(f"  Resumed epoch {start_epoch} | best F1={best_f1:.4f}")

    if start_epoch >= EXPERT_EPOCHS:
        print(f"  Already fully trained. Skipping.")
        return best_f1

    optimizer = get_optimizer(model, stage)
    # Cosine annealing with warmup (handled manually in loop)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EXPERT_EPOCHS - max(start_epoch, 1), eta_min=LR_MIN)

    for epoch in range(start_epoch, EXPERT_EPOCHS):
        epoch_start = time.perf_counter()
        if epoch == STAGE1_EPOCHS and stage == 1:
            stage = 2
            print(f"  -> Stage 2: unfreeze all layers")
            unfreeze_all(model)
            optimizer = get_optimizer(model, stage)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=STAGE2_EPOCHS, eta_min=LR_MIN)

        # Warmup: linear increase for first WARMUP_EPOCHS
        if epoch < WARMUP_EPOCHS:
            warmup_factor = (epoch + 1) / WARMUP_EPOCHS
            for param_group in optimizer.param_groups:
                param_group['lr'] = param_group['lr'] * warmup_factor

        model.train()
        optimizer.zero_grad()
        train_loss, train_correct, n = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"E{epoch+1:02d}", leave=False)
        for step, (imgs, lbls) in enumerate(pbar):
            imgs = imgs.to(DEVICE, non_blocking=True)
            lbls = lbls.to(DEVICE, non_blocking=True)
            if stage == 2 and np.random.random() < 0.5:
                imgs, la, lb, lam = mixup_batch(imgs, lbls)
                with torch.amp.autocast("cuda"):
                    out = compiled(imgs)
                    loss = lam * criterion(out, la) + (1 - lam) * criterion(out, lb)
            else:
                with torch.amp.autocast("cuda"):
                    out = compiled(imgs)
                    loss = criterion(out, lbls)
            loss = loss / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                ema.update()
            train_loss += loss.item() * GRAD_ACCUM_STEPS * len(imgs)
            train_correct += (out.argmax(1) == lbls).sum().item()
            n += len(imgs)
            pbar.set_postfix(loss=f"{train_loss/n:.3f}", acc=f"{train_correct/n:.3f}")

        scheduler.step()
        train_loss /= n
        train_acc = train_correct / n
        val_loss, val_acc, val_f1, _, _ = evaluate_expert(model, ema, val_loader)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_f1"].append(val_f1)
        elapsed = time.perf_counter() - epoch_start
        print(f"  E{epoch+1:02d} | trn_loss={train_loss:.4f} trn_acc={train_acc:.4f} | val_acc={val_acc:.4f} val_F1={val_f1:.4f} | {elapsed:.0f}s")

        if val_f1 > best_f1:
            best_f1 = val_f1
            patience = 0
            save_checkpoint({"model_state": model.state_dict(), "ema_state": ema.state_dict(),
                             "epoch": epoch, "best_f1": best_f1, "stage": stage, "history": dict(history)}, ckpt_best)
        else:
            patience += 1
            if patience >= EARLY_STOP_PAT:
                print(f"  Early stop at epoch {epoch+1} (patience={EARLY_STOP_PAT})")
                break
        save_checkpoint({"model_state": model.state_dict(), "ema_state": ema.state_dict(),
                         "epoch": epoch, "best_f1": best_f1, "stage": stage, "history": dict(history)}, ckpt_latest)
    print(f"  Best val F1: {best_f1:.4f}")
    return best_f1


## 🚀 Section 7 — Train All Experts (Target <1 min each)

In [ ]:
expert_results = {}
with timer("Expert training (all)"):
    for expert_name, builder in EXPERT_BUILDERS.items():
        with timer(f"Expert: {expert_name}"):
            model = builder()
            f1 = train_expert(expert_name, model, resume=True)
            expert_results[expert_name] = f1
            del model
            gc.collect()
            torch.cuda.empty_cache()
print("\n" + "="*50)
print("Expert F1 scores:")
for name, f1 in expert_results.items():
    print(f"  {name:<20}: {f1:.4f}")


## 🧠 Section 8 — Gating Network & MoE Classifier

In [ ]:
class ExpertFeatureExtractor(nn.Module):
    def __init__(self, model, feature_dim):
        super().__init__()
        self.model = model
        self.feature_dim = feature_dim
    def forward(self, x):
        if hasattr(self.model, "layer4"):
            feats = nn.Sequential(*list(self.model.children())[:-1])(x).flatten(1)
            logits = self.model.fc(feats)
        elif hasattr(self.model, "classifier"):
            feats = self.model.features(x)
            feats = self.model.avgpool(feats).flatten(1)
            logits = self.model.classifier(feats)
        elif hasattr(self.model, "blocks"):
            feats = self.model.forward_features(x)
            if feats.dim() == 3:
                feats = feats[:, 0]
            logits = self.model.head(feats)
        else:
            feats = self.model.backbone(x)
            if feats.dim() == 4:
                feats = nn.AdaptiveAvgPool2d(1)(feats).flatten(1)
            logits = self.model.head(feats)
        return feats, logits

class GatingNetwork(nn.Module):
    def __init__(self, expert_dims, hidden=GATE_HIDDEN, n_experts=NUM_EXPERTS):
        super().__init__()
        in_dim = sum(expert_dims)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(hidden // 2, n_experts),
        )
        self.n_experts = n_experts
    def forward(self, features_list):
        x = torch.cat(features_list, dim=1)
        return F.softmax(self.net(x), dim=1)
    def load_balance_loss(self, weights):
        f = weights.mean(0)
        return self.n_experts * (f * f).sum()

class MoEDamageClassifier(nn.Module):
    def __init__(self, expert_extractors, gate):
        super().__init__()
        self.experts = nn.ModuleList(expert_extractors)
        self.gate = gate
    def forward(self, x):
        all_feats, all_logits = [], []
        for expert in self.experts:
            feats, logits = expert(x)
            all_feats.append(feats)
            all_logits.append(logits)
        gate_weights = self.gate(all_feats)
        fused = (gate_weights.unsqueeze(-1) * torch.stack(all_logits, dim=1)).sum(dim=1)
        return fused, gate_weights, all_logits
    @torch.no_grad()
    def predict(self, x, conf_threshold=CONF_THRESHOLD):
        self.eval()
        fused, gate_w, expert_logits = self(x)
        probs = F.softmax(fused, dim=1)
        conf, pred_idx = probs.max(1)
        gate_max = gate_w.max(1).values
        used_fallback = (gate_max < conf_threshold).any().item()
        predicted_class = CLASS_NAMES[pred_idx[0].item()]
        criticality = CRITICALITY_MAP[predicted_class]
        expert_names = list(EXPERT_BUILDERS.keys())
        per_expert_preds = []
        for i, logits in enumerate(expert_logits):
            p = F.softmax(logits, dim=1)
            c, ci = p.max(1)
            per_expert_preds.append({
                "expert": expert_names[i],
                "class": CLASS_NAMES[ci[0].item()],
                "confidence": float(c[0].item()),
                "gate_weight": float(gate_w[0, i].item()),
            })
        return {
            "predicted_class": predicted_class,
            "confidence": float(conf[0].item()),
            "criticality": criticality,
            "gate_weights": gate_w[0].cpu().tolist(),
            "per_expert": per_expert_preds,
            "used_fallback": used_fallback,
            "class_probs": {CLASS_NAMES[i]: float(p) for i, p in enumerate(probs[0].cpu().tolist())},
        }
print("Gating & MoE defined.")


## 📦 Section 9 — Assemble MoE

In [ ]:
def load_expert(name, builder):
    model = builder().to(DEVICE)
    ckpt = load_checkpoint(f"{name}_best.pth")
    if ckpt:
        model.load_state_dict(ckpt["model_state"])
        print(f"  Loaded {name} (F1={ckpt['best_f1']:.4f})")
    else:
        print(f"  WARNING: {name} weights not found — random init")
    model.eval()
    return model

loaded_experts = {}
for name, builder in EXPERT_BUILDERS.items():
    loaded_experts[name] = load_expert(name, builder)

# Feature dimensions (hardcoded for simplicity)
EXPERT_FEAT_DIMS = {"resnet50": 2048, "efficientnet_b4": 1792, "vit_b16": 768, "yolo_damage": 256}
extractors = []
for name in EXPERT_BUILDERS:
    model = loaded_experts[name]
    dim = EXPERT_FEAT_DIMS[name]
    extractor = ExpertFeatureExtractor(model, dim)
    extractor.eval()
    for p in extractor.parameters():
        p.requires_grad = False
    extractors.append(extractor)

gate = GatingNetwork(expert_dims=list(EXPERT_FEAT_DIMS.values()), hidden=GATE_HIDDEN, n_experts=NUM_EXPERTS).to(DEVICE)
moe = MoEDamageClassifier(expert_extractors=[e.to(DEVICE) for e in extractors], gate=gate).to(DEVICE)
print(f"\nMoE assembled. Total feature dim = {sum(EXPERT_FEAT_DIMS.values())}")


## 🔀 Section 10 — Train Gating Network

In [ ]:
with timer("Gate training"):
    gate_optimizer = torch.optim.AdamW(gate.parameters(), lr=GATE_LR, weight_decay=1e-4)
    gate_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(gate_optimizer, T_max=GATE_EPOCHS, eta_min=1e-6)
    ce_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    gate_scaler = torch.amp.GradScaler("cuda")
    LB_WEIGHT = 0.01

    gate_history = defaultdict(list)
    best_gate_f1 = 0.0
    gate_ckpt = load_checkpoint("gate_best.pth")
    start_epoch = 0
    if gate_ckpt:
        gate.load_state_dict(gate_ckpt["gate_state"])
        best_gate_f1 = gate_ckpt.get("best_f1", 0.0)
        start_epoch = gate_ckpt.get("epoch", 0) + 1
        gate_history = defaultdict(list, gate_ckpt.get("history", {}))

    for epoch in range(start_epoch, GATE_EPOCHS):
        moe.train()
        for e in moe.experts: e.eval()
        total_loss, correct, n = 0.0, 0, 0
        for imgs, lbls in tqdm(train_loader, desc=f"Gate E{epoch+1:02d}", leave=False):
            imgs = imgs.to(DEVICE, non_blocking=True)
            lbls = lbls.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda"):
                fused, gate_weights, _ = moe(imgs)
                ce_loss = ce_criterion(fused, lbls)
                lb_loss = gate.load_balance_loss(gate_weights)
                loss = ce_loss + LB_WEIGHT * lb_loss
            gate_optimizer.zero_grad()
            gate_scaler.scale(loss).backward()
            gate_scaler.unscale_(gate_optimizer)
            torch.nn.utils.clip_grad_norm_(gate.parameters(), 1.0)
            gate_scaler.step(gate_optimizer)
            gate_scaler.update()
            total_loss += loss.item() * len(imgs)
            correct += (fused.argmax(1) == lbls).sum().item()
            n += len(imgs)
        gate_scheduler.step()
        moe.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    fused, _, _ = moe(imgs)
                val_preds.extend(fused.argmax(1).cpu().tolist())
                val_labels.extend(lbls.tolist())
        val_f1 = f1_score(val_labels, val_preds, average="weighted")
        val_acc = sum(p == l for p, l in zip(val_preds, val_labels)) / len(val_labels)
        gate_history["train_loss"].append(total_loss / n)
        gate_history["val_f1"].append(val_f1)
        print(f"  Gate E{epoch+1:02d} | loss={total_loss/n:.4f} | val_acc={val_acc:.4f} | val_F1={val_f1:.4f}")
        if val_f1 > best_gate_f1:
            best_gate_f1 = val_f1
            save_checkpoint({"gate_state": gate.state_dict(), "epoch": epoch, "best_f1": best_gate_f1,
                             "history": dict(gate_history)}, "gate_best.pth")
    print(f"\nBest gate F1: {best_gate_f1:.4f}")


## 📊 Section 11 — Evaluation, Per‑Expert Comparison & Grad‑CAM Heatmaps

In [ ]:
with timer("Evaluation"):
    ckpt = load_checkpoint("gate_best.pth")
    if ckpt:
        gate.load_state_dict(ckpt["gate_state"])
    moe.eval()

    all_preds, all_labels, all_confidences = [], [], []
    gate_weight_log = []
    with torch.no_grad():
        for imgs, lbls in tqdm(test_loader, desc="MoE Test"):
            imgs = imgs.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda"):
                fused, gate_weights, _ = moe(imgs)
            probs = F.softmax(fused, dim=1)
            confs, preds = probs.max(1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(lbls.tolist())
            all_confidences.extend(confs.cpu().tolist())
            gate_weight_log.extend(gate_weights.cpu().tolist())

    test_f1 = f1_score(all_labels, all_preds, average="weighted")
    test_acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
    print(f"\nMoE Test Accuracy      : {test_acc:.4f}")
    print(f"MoE Test F1 (weighted) : {test_f1:.4f}\n")
    print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

    # Per‑expert evaluation
    print("\nPer‑Expert Individual Test Performance:")
    expert_test_results = {}
    for name, extractor in zip(EXPERT_BUILDERS.keys(), moe.experts):
        ex_preds, ex_labels = [], []
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    _, logits = extractor(imgs)
                ex_preds.extend(logits.argmax(1).cpu().tolist())
                ex_labels.extend(lbls.tolist())
        ex_f1 = f1_score(ex_labels, ex_preds, average="weighted")
        ex_acc = sum(p == l for p, l in zip(ex_preds, ex_labels)) / len(ex_labels)
        expert_test_results[name] = {"f1": ex_f1, "acc": ex_acc}
        print(f"  {name:<20}: acc={ex_acc:.4f}  F1={ex_f1:.4f}")
    print(f"  {'MoE Ensemble':<20}: acc={test_acc:.4f}  F1={test_f1:.4f}")

    # Plots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    cm = confusion_matrix(all_labels, all_preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0,0])
    axes[0,0].set_title("MoE Confusion Matrix", fontweight="bold")
    axes[0,1].boxplot(np.array(gate_weight_log), labels=list(EXPERT_BUILDERS.keys()), patch_artist=True)
    axes[0,1].set_title("Gate Weight Distribution")
    axes[0,1].axhline(1/NUM_EXPERTS, color="r", linestyle="--", label="uniform")
    axes[0,1].legend()
    names = list(expert_test_results.keys()) + ["MoE"]
    f1s = [v["f1"] for v in expert_test_results.values()] + [test_f1]
    axes[1,0].bar(names, f1s, color=["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"])
    axes[1,0].set_title("Test F1: Experts vs MoE")
    axes[1,0].set_ylabel("Weighted F1")
    axes[1,1].hist(all_confidences, bins=50, alpha=0.7, edgecolor="black")
    axes[1,1].axvline(CONF_THRESHOLD, color="r", linestyle="--", label=f"threshold={CONF_THRESHOLD}")
    axes[1,1].set_title("MoE Confidence Distribution")
    axes[1,1].legend()
    plt.tight_layout()
    plt.savefig(DIRS["outputs"] / "moe_evaluation.png", dpi=150)
    plt.show()

    # Grad‑CAM on a few validation samples
    print("\nGenerating Grad‑CAM heatmaps (highest‑weight expert)...")
    sample_paths = list((DATA_DIR / "val").rglob("*.jpg"))[:3]
    if not sample_paths:
        sample_paths = list((DATA_DIR / "val").rglob("*.png"))[:3]
    for img_path in sample_paths:
        img_pil = PILImage.open(img_path).convert("RGB")
        img_np = np.array(img_pil) / 255.0
        tensor = val_tfm(image=np.array(img_pil))["image"].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            fused, gate_w, _ = moe(tensor)
            best_expert_idx = gate_w.argmax().item()
            best_expert_name = list(EXPERT_BUILDERS.keys())[best_expert_idx]
            # Get the actual model for Grad‑CAM
            expert_model = moe.experts[best_expert_idx].model
            target_layer = None
            if hasattr(expert_model, "layer4"):          # ResNet
                target_layer = expert_model.layer4[-1]
            elif hasattr(expert_model, "blocks"):       # ViT
                target_layer = expert_model.blocks[-1].norm1
            else:                                         # EfficientNet
                target_layer = expert_model.features[-1]
            if target_layer:
                for p in expert_model.parameters():
                    p.requires_grad = True
                cam = GradCAM(model=expert_model, target_layers=[target_layer])
                grayscale_cam = cam(input_tensor=tensor, targets=None)[0]
                visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
                plt.figure(figsize=(8, 4))
                plt.subplot(1,2,1); plt.imshow(img_np); plt.title("Original"); plt.axis("off")
                plt.subplot(1,2,2); plt.imshow(visualization); plt.title(f"Grad‑CAM ({best_expert_name})"); plt.axis("off")
                plt.tight_layout()
                out_path = DIRS["gradcam"] / f"gradcam_{img_path.stem}.png"
                plt.savefig(out_path, bbox_inches="tight")
                print(f"  Saved {out_path}")
                plt.close()


## 🔁 Section 12 — Fallback Chain Validation

In [ ]:
def predict_single(pil_image, moe_model, conf_threshold=CONF_THRESHOLD):
    img = np.array(pil_image.convert("RGB"))
    tensor = val_tfm(image=img)["image"].unsqueeze(0).to(DEVICE)
    try:
        result = moe_model.predict(tensor, conf_threshold)
        result["source"] = "moe_gate" if not result["used_fallback"] else "ensemble_fallback"
        return result
    except Exception as e:
        print(f"  MoE inference failed: {e} — using mock fallback")
        return {
            "predicted_class": "Undamaged", "confidence": 0.0, "criticality": "STABLE",
            "gate_weights": [1/NUM_EXPERTS] * NUM_EXPERTS, "per_expert": [],
            "used_fallback": True, "source": "mock_fallback",
            "class_probs": {c: 1/NUM_CLASSES for c in CLASS_NAMES},
        }

val_imgs = list((DATA_DIR / "val").rglob("*.jpg"))[:5]
if not val_imgs:
    val_imgs = list((DATA_DIR / "val").rglob("*.png"))[:5]
print(f"{'Ground Truth':<20} {'Predicted':<18} {'Conf':>6}  {'Criticality':<12} Source")
print("-" * 80)
for p in val_imgs:
    img = PILImage.open(p)
    res = predict_single(img, moe)
    print(f"  {p.parent.name:<18} -> {res['predicted_class']:<18} {res['confidence']:.3f}  {res['criticality']:<12}  {res['source']}")


## 💾 Section 13 — Export Weights for Web App

In [ ]:
import zipfile
def zip_weights():
    zip_path = BASE_DIR / "moe_weights.zip"
    weight_files = list(WEIGHTS_DIR.glob("*.pth"))
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in weight_files:
            zf.write(f, f.name)
            print(f"  + {f.name}  ({f.stat().st_size/1e6:.1f} MB)")
    print(f"\nSaved -> {zip_path} ({zip_path.stat().st_size/1e6:.1f} MB)")
    return zip_path
zip_weights()

manifest = {
    "resnet50": "resnet50_best.pth", "efficientnet_b4": "efficientnet_b4_best.pth",
    "vit_b16": "vit_b16_best.pth", "yolo_damage": "yolo_damage_best.pth",
    "gate": "gate_best.pth", "class_names": CLASS_NAMES, "criticality_map": CRITICALITY_MAP,
    "conf_threshold": CONF_THRESHOLD, "expert_feat_dims": EXPERT_FEAT_DIMS,
    "gate_hidden": GATE_HIDDEN,
}
with open(WEIGHTS_DIR / "moe_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("\nManifest:")
print(json.dumps(manifest, indent=2))
print("\nDrop into your web app MODEL_WEIGHTS_DIR and update model_registry.py")


## ⏱️ Section 14 — Training Summary

In [ ]:
print("\n" + "="*60)
print("  TRAINING SUMMARY (Ultra‑Fast Mode)")
print("="*60)
print("\nTiming Breakdown:")
total = 0.0
for name, secs in sorted(TIMINGS.items(), key=lambda x: -x[1]):
    m, s = divmod(secs, 60)
    total += secs
    print(f"  {name:<35} {int(m):3d}m {s:5.1f}s")
h, rem = divmod(total, 3600); m, s = divmod(rem, 60)
print(f"  {'TOTAL':<35} {int(h)}h {int(m)}m {s:.0f}s")
print(f"\n  GPU hours used: {total / 3600:.2f}")
print("\nModel Performance (Test Set):")
print(f"  {'Model':<20} {'F1':>8}")
print(f"  {'-'*28}")
for name in EXPERT_BUILDERS:
    ckpt = load_checkpoint(f"{name}_best.pth")
    f1 = ckpt["best_f1"] if ckpt else 0.0
    print(f"  {name:<20} {f1:>8.4f}")
print(f"  {'MoE Ensemble':<20} {test_f1:>8.4f}")
print("\n✅ Notebook complete. Download moe_weights.zip from /kaggle/working.")


## 💡 Section 15 — Extension Ideas
- **Severity score** – regression head for 0–1 continuous damage.
- **Temporal change** – compare embeddings across time.
- **MC Dropout uncertainty** – flag low‑confidence samples.
- **Priority queue** – rank by criticality × heritage value.
- **Few‑shot adaptation** – fine‑tune on 5–10 new temple examples.
